In [13]:
import chromadb

# 1. 連接到你的資料庫資料夾
client = chromadb.PersistentClient(path="./hiwin_vector_db")

# 2. 列出該資料夾下所有的 Collection 名稱
collections = client.list_collections()

print("--- 目前資料庫中的所有 Collection ---")
for col in collections:
    print(f"名稱: {col.name}")

--- 目前資料庫中的所有 Collection ---
名稱: hiwin_specs
名稱: hiwin_manual


In [14]:
import chromadb
from chromadb.utils import embedding_functions

# 1. 重新呼叫翻譯官 
emb_fn = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name="BAAI/bge-m3",
    device="cuda"  # 確保調用你的 RTX 4060
)

# 2. 連接到硬碟裡的資料庫
client = chromadb.PersistentClient(path="./hiwin_vector_db")

# 3. 取得當初存入 Chunks 的 Collection
manual_collection = client.get_collection(name = "hiwin_manual", embedding_function = emb_fn)
spec_collection = client.get_collection(name = "hiwin_specs", embedding_function = emb_fn)


print("成功連接向量資料庫，Chunks 已準備好被檢索！")

成功連接向量資料庫，Chunks 已準備好被檢索！


In [ ]:
import ollama

# 定義專業知識字典
SERIES_INFO = {
    "FDC": "雙螺帽設計，具備極高的軸向剛性與預壓穩定性，專為重負荷精密工具機設計。",
    "FSW": "小法蘭單螺帽設計，體積精簡，適合安裝空間受限的自動化設備。",
    "FSV": "標準單螺帽型，具備優異的傳動效率與流暢度，是自動化產業最泛用的標準件。",
    "RSI": "旋轉螺帽設計，適合絲槓固定、螺帽旋轉的機構，能有效抑制長行程下的振動。",
    "FSI": "內循環設計，螺帽外徑小，運轉安靜，適合小型精密設備。"
}

def get_expert_advice(user_query, calc_result, use_rag=True):
    """
    混合檢索架構：同時檢索技術手冊 (Manual) 與 產品規格 (Specs)
    """
    rag_context = ""
    rag_status_msg = ""

    # --- RAG 混合檢索邏輯 ---
    if use_rag:
        try:
            # A. 檢索【技術手冊】(Manual Chunks): 找潤滑、安裝、原理、壽命
            manual_res = manual_collection.query(query_texts=[user_query], n_results=2)
            manual_text = "\n【技術手冊參考資料】：\n" + "\n".join(manual_res['documents'][0])
            
            # B. 檢索【產品規格】(Specs Documents): 找替代型號、詳細尺寸、參數對比
            spec_res = spec_collection.query(query_texts=[user_query], n_results=3)
            spec_text = "\n【相似型號規格參考】：\n" + "\n".join(spec_res['documents'][0])
            
            # 整合兩路檢索結果
            rag_context = f"{spec_text}\n{manual_text}"
            rag_status_msg = "\n(系統提示：已完成混合檢索 - 參考手冊與規格表)\n"
            
        except Exception as e:
            rag_context = f"\n(系統提示：資料庫檢索失敗: {e})\n"
    
    # --- 數據準備 (目前計算出的最優解) ---
    series = calc_result.get('series', '標準')
    model = calc_result.get('model', '未知')
    feature = SERIES_INFO.get(series, "HIWIN 精密傳動元件。")
    
    spec_context = f"""
    【目前推薦型號數據】
    - 推薦系列：{series} ({feature})
    - 具體型號：{model}
    - 物理參數：公稱外徑 {calc_result['dia']}mm, 導程 {calc_result['lead']}mm
    - 動負荷能力：{calc_result['dynamic_load']} kgf
    """
    
    # --- 組合最終 Prompt ---
    # 這裡我們明確區分「目前數據」與「參考資料」，幫助 LLM 進行對比分析
    prompt = f"""
    你是一位專業的 HIWIN 技術支援工程師，請根據提供的【目前推薦型號數據】與【參考資料】來回答提問。
    回答時請結合產品的物理特性（如外徑、負荷）與系列優點（如剛性、空間利用)，請用繁體中文回答。
       
    當使用者詢問關於空間、尺寸或替代型號時，請優先參考「相似型號規格」進行對比。
    當使用者詢問關於安裝、保養或技術原理時，請參考「技術手冊參考資料」。

    {spec_context}
    
    {rag_context}
    
    使用者提問：{user_query}
    
    請用繁體中文回答，語氣專業且誠懇，並儘可能引用具體參數。
    """
    
    # --- 呼叫 Qwen 2.5 ---
    try:
        response = ollama.generate(
            model='qwen2.5:7b', 
            prompt=prompt,
            options={"temperature": 0.3} 
        )
        return response['response'] + rag_status_msg
    except Exception as e:
        return f"連線 Ollama 發生錯誤: {str(e)}"

In [17]:
#模型:Ollama qwen2.5:7b，無RAG回答

# 模擬你公式跑完後的結果 (Dictionary 格式)
calc_result = {
    "series": "FDC",
    "model": "40-12K5",
    "dia": 40.0,
    "lead": 12.0,
    "dynamic_load": 7430
}

# 測試提問
user_query = "螺帽直徑跟長度空間有限，是否有其他型號建議"

print("正在調用 Qwen 2.5:7b 進行分析...\n")
result = get_expert_advice(user_query, calc_result, use_rag = False)

from IPython.display import Markdown
Markdown(result) # 使用 Markdown 讓回答看起來更漂亮

正在調用 Qwen 2.5:7b 進行分析...



您好，

感謝您對HIWIN產品的關注。根據您的需求，我們可以考慮以下幾種替代型號，這些型號在物理特性上與40-12K5相近，但可能在外徑或導程上有不同的設計，以滿足您對於空間的需求。

1. **36-12K5**：此型號的公稱外徑為36.0mm，導程仍保持12.0mm。相比40-12K5，其外徑減少4mm，可能更適合於空間有限的情況下使用。同時，其動負荷能力仍可達7280 kgf，接近您目前推薦型號的動負荷能力。

2. **40-10K5**：此型號的公稱外徑為40.0mm，但導程減少至10.0mm。雖然外徑與您現有的型號相同，但較短的導程可能在某些應用中更易於安裝和使用。

3. **32-12K5**：此型號的公稱外徑為32.0mm，導程仍保持12.0mm。相比40-12K5，其外徑減少8mm，進一步減小了空間需求。動負荷能力為6790 kgf。

以上型號均採用FDC系列的雙螺帽設計，具有極高的軸向剛性與預壓穩定性，適合重負荷精密工具機應用。您可以根據具體安裝和使用環境來選擇最合適的型號。

如需進一步技術支援或詳細資料，請隨時聯繫我們。感謝您的信任！

敬上

In [16]:
#模型:Ollama qwen2.5:7b，有RAG回答

# 模擬你公式跑完後的結果 (Dictionary 格式)
calc_result = {
    "series": "FDC",
    "model": "40-12K5",
    "dia": 40.0,
    "lead": 12.0,
    "dynamic_load": 7430
}

# 測試提問
user_query = "螺帽直徑跟長度空間有限，是否有其他型號建議?"

print("正在調用 Qwen 2.5:7b 進行分析...\n")
result = get_expert_advice(user_query, calc_result, use_rag = True)

from IPython.display import Markdown
Markdown(result) # 使用 Markdown 讓回答看起來更漂亮

正在調用 Qwen 2.5:7b 進行分析...



根據您提供的【目前推薦型號數據】與【相似型號規格參考】，我們可以為您提供一些適合的替代型號建議。

首先，您現有的40-12K5型號具有公稱外徑40.0mm、導程12.0mm以及7430 kgf的動負荷能力。由於螺帽直徑和長度空間有限，我們可以考慮以下幾種替代方案：

1. **上銀 HIWIN 滾珠螺桿型號 32-8T4 (系列: RSI)**：此型號具有公稱外徑32mm、導程8.0mm的規格。雖然其動負荷能力較低，為2317 kgf，但對於空間限制較嚴格的情況下，可以考慮使用此型號。此外，該型號適合長行程且需高速旋轉螺帽的特殊機構。

2. **上銀 HIWIN 滾珠螺桿型號 45-10B1 (系列: FSW)**：此型號具有公稱外徑45mm、導程10.0mm的規格，動負荷能力為3116 kgf。相較於40-12K5型號，其外徑稍大，但動負荷能力有所提升。該型號結構輕巧省空間，適合小型自動化設備或精密儀器。

在選擇替代型號時，我們需要考慮您的應用需求和空間限制。根據【技術手冊參考資料】中的表4.10，可以得知不同外徑的滾珠螺桿所能達到的最大長度範圍。例如，公稱外徑40mm的C2級精度螺桿最大加工長度為1300mm。

建議您根據具體應用需求和空間限制進一步評估這些型號是否符合您的要求。如有更詳細的需求或疑問，歡迎隨時聯繫我們的技術支援團隊，我們將竭誠提供專業建議與支持。
(系統提示：已完成混合檢索 - 參考手冊與規格表)
